# Session 1: Fine-tuning a Transformer with Hugging Face

**Goal of today:** take a pre-trained model (DistilBERT) that knows English but knows nothing about our task, and teach it to classify movie reviews as POSITIVE or NEGATIVE.

We will do it two ways:

1. **Full fine-tuning** - update all ~67 million weights.
2. **LoRA (Parameter-Efficient Fine-Tuning)** - freeze the model, train ~1% extra weights.

Then we compare them on accuracy, time, GPU memory and disk size.

**Before you start (Colab):** `Runtime -> Change runtime type -> T4 GPU`. Fine-tuning on CPU will take hours.

**Flow of the notebook**

| Part | What we do |
|---|---|
| 1 | Setup and version check |
| 2 | Load and inspect the dataset |
| 3 | Tokenizer: what the model actually sees |
| 4 | Model: what is inside, what is new |
| 5 | One forward pass by hand (under the hood) |
| 6 | Full fine-tuning with `Trainer` |
| 7 | Evaluate and look at the mistakes |
| 8 | Save, reload, and use with `pipeline` |
| 9 | LoRA fine-tuning |
| 10 | Compare both approaches |
| 11 | Exercises |

**Student copy:** cells marked `TODO` are written together in class. The complete notebook is shared after the session.

## Part 1: Setup

We pin exact library versions. Hugging Face libraries change fast, and many online tutorials break because an argument got renamed (you will see two such renames today). Pinning means this notebook behaves the same next month.

We do **not** pin `torch` - Colab already has a GPU build of it installed.

In [ ]:
!pip install -q transformers==5.19.0 datasets==5.1.0 peft==0.21.2 accelerate==1.15.0 scikit-learn

Imports and a quick check: library versions and whether a GPU is visible.
If `GPU available` prints `False`, change the runtime type before going further.

In [ ]:
import os
import gc
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

import transformers
import datasets
import peft

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    pipeline,
    set_seed,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

print("transformers :", transformers.__version__)
print("datasets     :", datasets.__version__)
print("peft         :", peft.__version__)
print("torch        :", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name     :", torch.cuda.get_device_name(0))

# same random numbers every run -> results are comparable between runs
set_seed(42)

A small helper we will reuse: total size of a folder on disk in MB. We use it later to compare the saved full model vs the saved LoRA adapter.

In [ ]:
def folder_size_mb(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return round(total / (1024 * 1024), 2)

## Part 2: Load and inspect the dataset

**IMDB** - 50,000 movie reviews, each labelled `0` (negative) or `1` (positive). 25k for training, 25k for testing.

`load_dataset` downloads it from the Hugging Face Hub and caches it, so the second run is instant.

In [ ]:
raw = load_dataset("stanfordnlp/imdb")
raw

A `DatasetDict` is like a Python dict of tables: `train`, `test` (and `unsupervised`, which has no labels - we ignore it).

Let us look at what the label column really is, and at one row.

In [ ]:
print(raw["train"].features)
print()

label_names = raw["train"].features["label"].names
print("label names:", label_names)

sample = raw["train"][0]
print("label:", sample["label"], "->", label_names[sample["label"]])
print("text :", sample["text"][:500], "...")

**Why a subset?** Training on all 25k reviews takes ~25 minutes on a T4. For class we use 4,000 training and 1,000 test reviews - enough to get ~88% accuracy in a few minutes.

`shuffle` first is important: the IMDB train split is **sorted by label** (all negatives, then all positives). Taking the first 4,000 rows without shuffling would give us only negative reviews.

In [ ]:
train_raw = raw["train"].shuffle(seed=42).select(range(4000))
test_raw = raw["test"].shuffle(seed=42).select(range(1000))

# proof of the sorting problem: first 4000 rows without shuffling
print("unshuffled first 4000 labels:", pd.Series(raw["train"][:4000]["label"]).value_counts().to_dict())

# after shuffling, both classes are present in a balanced way
print("train label counts:", pd.Series(train_raw["label"]).value_counts().to_dict())
print("test  label counts:", pd.Series(test_raw["label"]).value_counts().to_dict())

How long are the reviews? This matters because the model has a maximum input length (512 tokens for DistilBERT). Here we count words; next part we count tokens.

In [ ]:
word_counts = pd.Series([len(t.split()) for t in train_raw["text"]])
print(word_counts.describe().round(1))

word_counts.plot.hist(bins=50, title="Words per review (train subset)")
plt.xlabel("words")
plt.show()

## Part 3: The tokenizer - what the model actually sees

A model cannot read text. It reads **numbers**. The tokenizer:

1. splits text into tokens (words or word-pieces),
2. maps each token to an ID from a fixed vocabulary,
3. adds special tokens the model was trained with (`[CLS]` at the start, `[SEP]` at the end),
4. builds an `attention_mask` (1 = real token, 0 = padding).

**Rule:** always load the tokenizer with the *same name* as the model. A different tokenizer = different IDs = garbage input.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(type(tokenizer).__name__)
print("vocabulary size:", tokenizer.vocab_size)
print("max length     :", tokenizer.model_max_length)
print("special tokens :", tokenizer.all_special_tokens)

Tokenize one sentence and look at every piece.

Notice:
- `uncased` model -> everything is lower-cased.
- Rare words get split into pieces; continuation pieces start with `##`.
- `[CLS]` (id 101) and `[SEP]` (id 102) are added automatically.

In [ ]:
text = "The movie was unbelievably boring!"
enc = tokenizer(text)

print("input_ids     :", enc["input_ids"])
print("attention_mask:", enc["attention_mask"])
print("tokens        :", tokenizer.convert_ids_to_tokens(enc["input_ids"]))
print("decoded back  :", tokenizer.decode(enc["input_ids"]))

Now count **tokens** (not words) for the training reviews. Tokens are always more than words, because of word-pieces and punctuation.

We use this to choose `max_length`. Anything longer gets truncated (cut). Longer `max_length` = more GPU memory and slower training, so we pick a value that covers most reviews without being wasteful.

**Library-change trap #0:** in `datasets` 5.x, `train_raw["text"]` is a lazy *Column* object, not a Python list. The tokenizer only accepts `str` or `list[str]`, so we wrap it in `list(...)`. Old tutorials skip this and fail with a `ValueError`.

In [ ]:
print(type(train_raw["text"]).__name__)   # Column, not list

token_counts = pd.Series([len(ids) for ids in tokenizer(list(train_raw["text"]))["input_ids"]])
print(token_counts.describe().round(1))

for limit in [128, 256, 384, 512]:
    covered = (token_counts <= limit).mean() * 100
    print(f"max_length={limit}: {covered:.1f}% of reviews fit without cutting")

We pick **256**. Around half the reviews fit fully (check the printed numbers), and for the longer ones the first 256 tokens usually already make the sentiment clear. 512 would be more accurate but roughly 2-3x slower.

Now tokenize the whole subset with `.map(batched=True)` - this sends 1,000 rows at a time to the tokenizer, which is much faster than one by one.

Note: we do **not** pad here. Padding happens later, per batch (see the data collator).

In [ ]:
MAX_LENGTH = 256

def tokenize(batch):
    # TODO: call the tokenizer on batch["text"]
    #       - cut long reviews at MAX_LENGTH (truncation)
    #       - do NOT pad here
    pass

train_tok = train_raw.map(tokenize, batched=True, remove_columns=["text"])
test_tok = test_raw.map(tokenize, batched=True, remove_columns=["text"])

print(train_tok)
print("lengths of first 5 rows:", [len(x) for x in train_tok[:5]["input_ids"]])

**Dynamic padding with a data collator.**

A GPU needs every row in a batch to have the same length. Two options:
- pad *everything* to 256 up front -> lots of wasted zeros for short reviews;
- pad *each batch* only up to its longest row -> what `DataCollatorWithPadding` does.

Below: three rows of different lengths become one tensor, padded to the longest of the three. Look at the zeros in `attention_mask`.

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

rows = [train_tok[i] for i in range(3)]
print("individual lengths:", [len(r["input_ids"]) for r in rows])

batch = data_collator(rows)
print("batch keys        :", list(batch.keys()))
print("input_ids shape   :", tuple(batch["input_ids"].shape))
print("attention_mask row lengths (real tokens):", batch["attention_mask"].sum(dim=1).tolist())
print("label tensor      :", batch["labels"])

One detail: our column is called `label`, but the batch key is `labels` (plural). The collator renames it, because the model's `forward()` expects `labels`. If you ever see an error about a missing `labels` argument, this is the first place to look.

## Part 4: The model - what is inside, what is new

`AutoModelForSequenceClassification` = pre-trained DistilBERT body **+ a brand-new classification head** on top.

- The body already understands language (trained on Wikipedia + books).
- The head is a small linear layer that maps the sentence representation to our 2 labels. It starts with **random weights**.

You will see a message saying some weights were *newly initialized*. That is expected - it is the head. It is exactly what we are about to train.

`id2label` / `label2id` store readable label names inside the model, so later predictions say `POSITIVE` instead of `LABEL_1`.

In [ ]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

Let us look inside: top-level blocks, total parameters, and the new head.

In [ ]:
for name, module in model.named_children():
    n_params = sum(p.numel() for p in module.parameters())
    print(f"{name:15s} {type(module).__name__:25s} {n_params:>12,} params")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\ntotal params    : {total_params:,}")
print(f"trainable params: {trainable_params:,}  (full fine-tuning trains everything)")

print("\nthe new head:")
print(model.pre_classifier)
print(model.classifier)

## Part 5: One forward pass by hand (under the hood)

Before using `Trainer`, let us do what it does internally for **one** batch:

1. send the batch through the model,
2. get **logits** (raw scores, one per class),
3. turn logits into probabilities with softmax,
4. compute the **loss** (cross-entropy) against the true labels.

`Trainer` repeats this thousands of times and, after each batch, adjusts the weights to reduce the loss (backpropagation + optimizer step).

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

batch = data_collator([train_tok[i] for i in range(4)])
batch = {k: v.to(device) for k, v in batch.items()}

model.eval()
with torch.no_grad():
    out = model(**batch)

print("logits shape:", tuple(out.logits.shape), "-> (batch size, number of classes)")
print("logits      :", out.logits.cpu().numpy().round(3))
print("probs       :", torch.softmax(out.logits, dim=-1).cpu().numpy().round(3))
print("true labels :", batch["labels"].tolist())
print("model loss  :", round(out.loss.item(), 4))

# the same loss, computed ourselves -> proves what the model does internally
manual_loss = F.cross_entropy(out.logits, batch["labels"])
print("manual loss :", round(manual_loss.item(), 4))

Probabilities are close to 0.5/0.5 and the loss is close to **0.693** (= ln 2). That is the loss of a model that is just guessing between 2 classes - which makes sense, the head is random.

After training we expect confident probabilities and a much lower loss.

## Part 6: Full fine-tuning with `Trainer`

### 6.1 Metrics

`Trainer` gives us the loss automatically. For humans, accuracy and F1 are easier to read. `compute_metrics` receives the logits and true labels for the whole eval set and returns a dict.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

### 6.2 Training arguments

| Argument | Value | Meaning |
|---|---|---|
| `num_train_epochs` | 2 | passes over the training data |
| `per_device_train_batch_size` | 16 | reviews per weight update |
| `learning_rate` | 2e-5 | small step size; large values destroy the pre-trained knowledge |
| `weight_decay` | 0.01 | mild regularisation against overfitting |
| `eval_strategy` | "epoch" | evaluate on the test set after every epoch |
| `save_strategy` | "epoch" | save a checkpoint after every epoch |
| `load_best_model_at_end` | True | keep the checkpoint with the best `metric_for_best_model` |
| `fp16` | True on GPU | half-precision math: faster and uses less memory on a T4 |
| `report_to` | "none" | do not try to log to Weights & Biases |

**Library-change trap #1:** older tutorials write `evaluation_strategy=`. It was renamed to `eval_strategy`; the old name now raises an error.

In [ ]:
# TODO: create TrainingArguments using the table above
#       output_dir="checkpoints_full", 2 epochs, batch size 16 (eval 32),
#       learning rate 2e-5, weight decay 0.01, evaluate + save every epoch,
#       keep the best model by "f1", log every 25 steps,
#       fp16 only if a GPU is available, report_to="none", seed 42
full_args = None

### 6.3 The Trainer

`Trainer` wires everything together: model, arguments, data, collator, metrics.

**Library-change trap #2:** older tutorials pass `tokenizer=tokenizer`. In current versions this argument is `processing_class=tokenizer` (it can also be an image processor, hence the general name).

We evaluate **before** training to get a baseline - expect ~50% accuracy (random guessing).

In [ ]:
# TODO: create the Trainer with model, args, train/eval datasets,
#       processing_class, data_collator and compute_metrics
full_trainer = None

baseline = full_trainer.evaluate()
print({k: round(v, 4) for k, v in baseline.items() if k in ["eval_loss", "eval_accuracy", "eval_f1"]})

### 6.4 Train

We also record wall-clock time and peak GPU memory, so we can compare with LoRA later. On a T4 this takes roughly 4-6 minutes.

In [ ]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start = time.time()
full_trainer.train()
full_time = time.time() - start

full_peak_mem = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else float("nan")
print(f"training time : {full_time / 60:.1f} min")
print(f"peak GPU mem  : {full_peak_mem:.2f} GB")

### 6.5 Training curve

`trainer.state.log_history` is a list of every log entry (training loss every 25 steps, eval metrics every epoch). Plotting it is the quickest way to see if training worked:
- training loss should go **down**,
- eval loss going **up** while training loss goes down = overfitting.

In [ ]:
history = pd.DataFrame(full_trainer.state.log_history)

train_curve = history.dropna(subset=["loss"])
eval_curve = history.dropna(subset=["eval_loss"])

plt.plot(train_curve["step"], train_curve["loss"], label="training loss")
plt.plot(eval_curve["step"], eval_curve["eval_loss"], "o-", label="eval loss")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Full fine-tuning")
plt.legend()
plt.show()

eval_curve[["epoch", "eval_loss", "eval_accuracy", "eval_f1"]].round(4)

## Part 7: Evaluate and look at the mistakes

Accuracy is one number. A confusion matrix shows **which** mistakes the model makes, and reading actual wrong predictions tells you **why**.

In [ ]:
full_metrics = full_trainer.evaluate()
print({k: round(v, 4) for k, v in full_metrics.items() if k in ["eval_loss", "eval_accuracy", "eval_f1"]})

pred_output = full_trainer.predict(test_tok)
preds = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

print()
print(classification_report(labels, preds, target_names=["NEGATIVE", "POSITIVE"], digits=3))

ConfusionMatrixDisplay.from_predictions(labels, preds, display_labels=["NEGATIVE", "POSITIVE"], cmap="Blues")
plt.title("Full fine-tuning - test set")
plt.show()

Now the interesting part: reviews the model got wrong, with its confidence. Typical reasons: sarcasm, mixed reviews ("great acting, terrible story"), or the sentiment appears only after the 256-token cut.

In [ ]:
probs = torch.softmax(torch.tensor(pred_output.predictions), dim=-1).numpy()
wrong = np.where(preds != labels)[0]
print(f"{len(wrong)} wrong out of {len(labels)}\n")

for i in wrong[:3]:
    print(f"true: {id2label[labels[i]]:8s} | predicted: {id2label[preds[i]]:8s} | confidence: {probs[i].max():.2f}")
    print(test_raw[int(i)]["text"][:400], "...")
    print("-" * 80)

## Part 8: Save, reload, and use with `pipeline`

`save_model` writes the weights (`model.safetensors`) and config; saving the tokenizer next to it makes the folder self-contained - anyone can load it with one line.

In [ ]:
FULL_DIR = "imdb-distilbert-full"
full_trainer.save_model(FULL_DIR)
tokenizer.save_pretrained(FULL_DIR)

print(sorted(os.listdir(FULL_DIR)))
print("size on disk:", folder_size_mb(FULL_DIR), "MB")

Reload from the folder with `pipeline` - the same `pipeline` you used with ready-made models, now pointing at **your** model. `top_k=None` returns the score for every label.

Try a few tricky sentences: negation, sarcasm, mixed opinion.

In [ ]:
classifier = pipeline("text-classification", model=FULL_DIR, top_k=None, device=0 if torch.cuda.is_available() else -1)

reviews = [
    "An absolute masterpiece. I would watch it again tomorrow.",
    "Two hours of my life I will never get back.",
    "Not bad at all, actually quite enjoyable.",
    "Oh great, another superhero movie. Just what the world needed.",
    "The acting was brilliant but the story made no sense.",
]

for r, scores in zip(reviews, classifier(reviews)):
    best = max(scores, key=lambda s: s["score"])
    print(f"{best['label']:8s} {best['score']:.3f} | {r}")

**What `pipeline` does under the hood** = tokenizer -> model -> softmax -> map index to label name. Same three steps by hand:

In [ ]:
loaded_model = AutoModelForSequenceClassification.from_pretrained(FULL_DIR).to(device).eval()

inputs = tokenizer(reviews[0], return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
with torch.no_grad():
    logits = loaded_model(**inputs).logits

# TODO: convert logits to probabilities, pick the best index,
#       and print its label name using loaded_model.config.id2label

## Part 9: LoRA - fine-tuning by training only ~1% of the weights

**The problem:** full fine-tuning updates every weight. For DistilBERT (67M) that is fine. For a 7-billion-parameter LLM it needs multiple large GPUs, and every fine-tuned version is another 14 GB file.

**The LoRA idea:**
- **Freeze** the original weight matrix `W` (it is never changed).
- Next to it, add two small matrices `A` and `B`. Their product `B x A` is the *change* we want: `W_new = W + B x A`.
- For a 768 x 768 matrix (590k numbers), with rank `r = 8`: `A` is 8 x 768 and `B` is 768 x 8 -> only ~12k numbers to train.

**Analogy:** instead of rewriting a 500-page book, you add sticky notes on a few pages. The book stays untouched; you can peel the notes off, or keep different sets of notes for different purposes.

First we free the GPU memory used by the full model.

In [ ]:
del model, full_trainer, loaded_model, classifier
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory in use now: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Load a **fresh** pre-trained model (not the one we just trained) and wrap it with LoRA.

| Argument | Value | Meaning |
|---|---|---|
| `task_type` | `SEQ_CLS` | tells PEFT this is sequence classification, so it also keeps the new head trainable |
| `r` | 8 | rank = size of the small matrices; higher = more capacity, more params |
| `lora_alpha` | 16 | scaling of the LoRA update (common rule: 2 x r) |
| `lora_dropout` | 0.1 | dropout on the LoRA path |
| `target_modules` | `["q_lin", "v_lin"]` | which layers get LoRA - DistilBERT names its query/value projections `q_lin`, `v_lin` |

`target_modules` names differ per model family (BERT: `query`, `value`; Llama: `q_proj`, `v_proj`). `print(model)` shows the names.

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id
)

# TODO: create a LoraConfig using the table above,
#       then wrap base_model with get_peft_model
lora_config = None
lora_model = None

lora_model.print_trainable_parameters()

Under the hood: what did PEFT change? One attention layer before and after - the original `Linear` is now `base_layer`, with `lora_A` and `lora_B` next to it.

In [ ]:
print(lora_model.base_model.model.distilbert.transformer.layer[0].attention.q_lin)

Which parameters are trainable now? Group them by type. Expect LoRA `A`/`B` matrices in 6 layers x 2 targets, plus the classification head.

In [ ]:
groups = {}
for name, p in lora_model.named_parameters():
    if not p.requires_grad:
        continue
    if "lora_" in name:
        key = "LoRA A/B matrices"
    elif "classifier" in name:
        key = "classification head"
    else:
        key = "other"
    groups[key] = groups.get(key, 0) + p.numel()

for k, v in groups.items():
    print(f"{k:22s} {v:>10,}")

Train with the same data and epochs. One change: **learning rate 2e-4** (10x higher). LoRA trains far fewer weights, starting from zero, so it needs bigger steps.

We skip saving checkpoints here to keep the run short.

In [ ]:
lora_args = TrainingArguments(
    output_dir="checkpoints_lora",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=25,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

lora_trainer = Trainer(
    model=lora_model,
    args=lora_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start = time.time()
lora_trainer.train()
lora_time = time.time() - start

lora_peak_mem = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else float("nan")
lora_metrics = lora_trainer.evaluate()

print(f"training time : {lora_time / 60:.1f} min")
print(f"peak GPU mem  : {lora_peak_mem:.2f} GB")
print({k: round(v, 4) for k, v in lora_metrics.items() if k in ["eval_loss", "eval_accuracy", "eval_f1"]})

Save the LoRA adapter. Only the small trained part is saved - not the 67M frozen weights. Compare the size with the full model folder.

In [ ]:
LORA_DIR = "imdb-distilbert-lora-adapter"
lora_model.save_pretrained(LORA_DIR)

print(sorted(os.listdir(LORA_DIR)))
print("adapter size   :", folder_size_mb(LORA_DIR), "MB")
print("full model size:", folder_size_mb(FULL_DIR), "MB")

To use the adapter later: load the original base model, attach the adapter, and optionally **merge** it into the base weights (`W + B x A` computed once). A merged model is a normal transformers model - no PEFT needed at inference and no extra latency.

In [ ]:
fresh_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id
)
with_adapter = PeftModel.from_pretrained(fresh_base, LORA_DIR)
merged_model = with_adapter.merge_and_unload()
print("merged model type:", type(merged_model).__name__)

lora_classifier = pipeline(
    "text-classification", model=merged_model, tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)
for r, res in zip(reviews, lora_classifier(reviews)):
    print(f"{res['label']:8s} {res['score']:.3f} | {r}")

## Part 10: Full fine-tuning vs LoRA

In [ ]:
comparison = pd.DataFrame([
    {
        "method": "Full fine-tuning",
        "trainable params": f"{total_params:,}",
        "train time (min)": round(full_time / 60, 1),
        "peak GPU mem (GB)": round(full_peak_mem, 2),
        "accuracy": round(full_metrics["eval_accuracy"], 4),
        "f1": round(full_metrics["eval_f1"], 4),
        "saved size (MB)": folder_size_mb(FULL_DIR),
    },
    {
        "method": "LoRA (r=8)",
        "trainable params": f"{sum(groups.values()):,}",
        "train time (min)": round(lora_time / 60, 1),
        "peak GPU mem (GB)": round(lora_peak_mem, 2),
        "accuracy": round(lora_metrics["eval_accuracy"], 4),
        "f1": round(lora_metrics["eval_f1"], 4),
        "saved size (MB)": folder_size_mb(LORA_DIR),
    },
])
comparison

**How to read this**

- Accuracy: LoRA usually lands within 1-3 points of full fine-tuning.
- Saved size: the biggest difference. One base model + many small adapters (one per task or client) instead of many full copies.
- Time and memory: savings exist but are modest on a small model like DistilBERT, because the forward pass still goes through all 67M frozen weights. On billion-parameter LLMs the memory saving is what makes fine-tuning possible on one GPU at all (and QLoRA - LoRA on a 4-bit quantized model - pushes this further).

**When to fine-tune at all?**

| Need | Usually the right tool |
|---|---|
| Model must know facts from your documents | RAG (Sessions 3-4) |
| Specific output format or behaviour | Prompting + structured output (Session 2) |
| Fixed label set, lots of labelled data, low cost per prediction | Fine-tune a small model (today) |
| Consistent style / domain language an LLM keeps missing | LoRA fine-tune an LLM |

Fine-tuning teaches **behaviour and patterns**, not reliable facts.

## Part 11: Exercises

1. **Easy** - Change `MAX_LENGTH` to 128 and retrain the full model. What happens to time and accuracy? Why?
2. **Medium** - Retrain LoRA with `r=2` and with `r=32`. Fill a small table: trainable params, accuracy, adapter size.
3. **Challenge** - Fine-tune on a different dataset: `fancyzhx/ag_news` (4 news categories: World, Sports, Business, Sci/Tech). What must change? (hint: `num_labels`, `id2label`, the label names, and the `f1_score` average)

**Optional - share your model on the Hugging Face Hub** (needs a free HF account and a write token):

```python
from huggingface_hub import notebook_login
notebook_login()
loaded = AutoModelForSequenceClassification.from_pretrained(FULL_DIR)
loaded.push_to_hub("your-username/imdb-distilbert-full")
tokenizer.push_to_hub("your-username/imdb-distilbert-full")
```